# Die render pipeline

Renders a die's spritesheets from the CC0 model in `assets/Dice D20 .../Dices blendswap.blend`.

Everything here is a thin wrapper over `pipeline.py`, so anything you can do in the notebook
you can also do from a shell:

```sh
python tools/dice-render/pipeline.py d20 --run --install --scene
```

**Opening this notebook does nothing.** Cells 1–3 only look at what is already on disk.
The render is cell 5, and it asks before starting.

| cell | |
|---|---|
| 1 | find the repository, Blender and the source blend |
| 2 | pick the die and the options |
| 3 | what is already rendered |
| 4 | one clip, to look at before committing to the rest |
| 5 | the full run |
| 6 | preview any clip as a strip of frames |
| 7 | install into `assets/`, regenerate the scene, validate |

A d20 is 22 clips and about 45 minutes. It renders one clip at a time and can be
interrupted: re-running picks up where it stopped.

## 1. Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / 'project.godot').is_file() and REPO != REPO.parent:
    REPO = REPO.parent
if not (REPO / 'project.godot').is_file():
    raise FileNotFoundError('run this from inside the repository')

TOOLS = REPO / 'tools' / 'dice-render'
sys.path.insert(0, str(TOOLS))
import dice_config, pipeline

BLENDER = Path(pipeline.find_blender())      # $BLENDER, or the known install paths
for label, path in [('repository', REPO), ('Blender', BLENDER),
                    ('source blend', Path(pipeline.BLEND))]:
    print('%-14s %s%s' % (label, path, '' if path.exists() else '   *** MISSING ***'))
print('%-14s %s' % ('dice', ', '.join(sorted(dice_config.DICE))))


def drive(*args, **kw):
    """Run pipeline.py, streaming its output into the notebook as it goes."""
    cmd = [sys.executable, '-u', str(TOOLS / 'pipeline.py')] + [str(a) for a in args]
    print('$ ' + ' '.join(cmd[2:]) + '\n')
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, errors='replace',
                         env={**os.environ, **kw.get('env', {})})
    for line in p.stdout:
        print(line, end='')
    return p.wait()

## 2. Choose the die

`DIE` must be a key of `DICE` in [`dice_config.py`](dice_config.py). Adding one is an entry
there — source object, face count, sheet prefix, scene path — not a change to any of the
scripts.

`RED` tints one face's glyph. `None` uses whatever the config says (the d6's 1; nothing on
the d20); a number overrides it for this run only, which is the way to look at a red 20
without editing the table and forgetting to put it back.

`KEEP_SUBFRAMES` leaves the 512px samples on disk instead of deleting each clip's once it
has been composited. Useful for looking at a single frame full size; about 2.2 GB for a d20
if you leave it on for a whole run.

In [ ]:
DIE            = 'd20'
RED            = None      # None = as configured, 20 = tint the 20, 'none' = tint nothing
KEEP_SUBFRAMES = False

CFG  = dice_config.die(DIE)
WORK = dice_config.work_dir(DIE)
OPTS = ([] if RED is None else ['--red', str(RED)]) + (['--keep'] if KEEP_SUBFRAMES else [])
CLIPS = [c[0] for c in pipeline.clips(CFG)]

print('%s: %s, %d faces -> %s' % (DIE, CFG['label'], CFG['faces'], CFG['scene']))
print('clips: %s' % ', '.join(CLIPS))
print('work:  %s' % WORK)

## 3. What is already on disk

Read-only. `composited` is the count that matters — a clip with all its frames is done and
will be skipped by the resuming run below.

In [ ]:
drive(DIE, '--status');

## 4. One clip first

Two minutes, and it catches the things that go wrong: a face-to-value table that is off, a
numeral lying on its side, a die framed too large for its cell. Cell 6 previews the result.

Change `CLIP` to any name from `CLIPS` — `'1'`, `'idle0'`.

In [ ]:
CLIP = '1'
drive(DIE, '--run', '--only', CLIP, '--keep', *OPTS);

## 5. The whole die

This is the long one. `--resume` skips clips that are already composited, so an interrupted
run continues rather than starting over — including the one clip from cell 4.

Nothing is written into the repository by this cell; the sheets land in the work directory.
Cell 7 installs them.

In [ ]:
if input('type RUN to render every clip of the %s: ' % DIE).strip() == 'RUN':
    drive(DIE, '--run', '--resume', *OPTS)
else:
    print('stopped; nothing rendered')

## 6. Look at it

The last frame of a numbered clip is the die at rest showing that number — the frame to
check a face-to-value table against.

In [ ]:
from PIL import Image
from IPython.display import display


def strip(clip, frames=None, zoom=2, sheets=None):
    """Frames of one clip, side by side, on the board's background colour."""
    sheet = Path(sheets or os.path.join(WORK, 'sheets')) / \
        os.path.basename(dice_config.sheet_path(CFG, clip))
    if not sheet.exists():
        raise FileNotFoundError('%s -- pack it first (cells 4, 5 or 7)' % sheet)
    im = Image.open(sheet).convert('RGBA')
    cell, cols = CFG['cell'], CFG['cols']
    n = CFG['idle_frames'] if clip in CFG['idles'] else CFG['roll_frames']
    frames = list(range(0, n, max(1, n // 8))) if frames is None else frames
    out = Image.new('RGBA', (cell * len(frames), cell), (0xD9, 0xC0, 0xA8, 0xFF))
    for i, f in enumerate(frames):
        r, c = divmod(f, cols)
        tile = im.crop((c * cell, r * cell, (c + 1) * cell, (r + 1) * cell))
        out.paste(tile, (i * cell, 0), tile)
    return out.resize((out.width * zoom, out.height * zoom), Image.LANCZOS)


def rest_row(values, zoom=2):
    """Each value's resting frame in a row: the face-to-value table at a glance."""
    last = CFG['roll_frames'] - 1
    tiles = [strip(str(v), [last], zoom=1) for v in values]
    out = Image.new('RGBA', (sum(t.width for t in tiles), tiles[0].height))
    x = 0
    for t in tiles:
        out.paste(t, (x, 0))
        x += t.width
    return out.resize((out.width * zoom, out.height * zoom), Image.LANCZOS)


display(strip('1'))                    # a throw, start to finish
display(strip('idle0'))                # the held tumble

In [ ]:
# Every face at rest. Read left to right: 1, 2, 3, ... Any number that is wrong or lying
# on its side is a face_values / face_twists entry to fix in dice_config.py.
for lo in range(1, CFG['faces'] + 1, 10):
    display(rest_row(range(lo, min(lo + 10, CFG['faces'] + 1))))

## 7. Install

**This is the cell that writes into the repository.** It copies the sheets into
`assets/dice/`, regenerates the die's `.tscn` from `dice_config.py` plus those sheets, and
checks every atlas region against the PNGs.

A die's scene is generated — do not hand-edit it (ROADMAP 8e).

Afterwards, add the scene to `DiceScenes` on the `GameManager` node in `scenes/game.tscn`
and it appears in the palette. Nothing else needs changing: `Dice.cs` counts a die's faces
off its own clips.

In [ ]:
if input('type INSTALL to write into assets/ and %s: ' % CFG['scene']).strip() == 'INSTALL':
    drive(DIE, '--run', '--resume', '--install', '--scene', *OPTS)
else:
    print('stopped; the repository is untouched')

## Housekeeping

The work directory holds the intermediates and is gitignored. With `KEEP_SUBFRAMES` off it
stays small; with it on, a finished d20 is a couple of gigabytes.

The sheets in `assets/` and the generated `.tscn` are the outputs — everything under
`build/` can be re-made. Uncomment the `rmtree` to clear it.

In [ ]:
import shutil

size = (sum(f.stat().st_size for f in Path(WORK).rglob('*') if f.is_file())
        if os.path.isdir(WORK) else 0)
print('%s: %.1f MB' % (WORK, size / 1024 / 1024))
# shutil.rmtree(WORK)